# EA2 — Despliegue y gobierno de una infraestructura de datos en la nube

**Big Data (ISD-25)** · Ingeniería de Software y Datos · IU Digital de Antioquia

| | |
| **Grupo** | (15) |
| **Integrantes** | (Adolfo Miguel Jimenez) |
| **Caso de estudio** | (Wanderbricks) |
| **Fecha de entrega** | domingo 6 de septiembre |
| **🎥 Enlace al video** | *(pegar aquí — 6 a 9 minutos, mínimo 3 minutos por integrante)* |

> ⚠️ **Antes de entregar:** verificar que el enlace del video abra desde una cuenta distinta a la propia.
> Un enlace inaccesible se califica como no entregado.

---
## 1. Contexto y problema

Wanderbricks es un marketplace de alquiler vacacional que genera un volumen constante de información a partir de la interacción de los clientes con la plataforma. Para este análisis nos enfocamos en tres ejes operativos principales: **usuarios**, **reservas** y **pagos**, además de los registros de navegación (**clickstream**).

**Necesidad del negocio y problema a resolver**
El objetivo central de negocio es comprender el comportamiento transaccional dentro de la plataforma, respondiendo específicamente a la pregunta: **¿qué métodos de pago utilizan los clientes y cuál es el método predominante?** 

Sin embargo, los datos en su estado original presentan desafíos importantes:
* Se encuentran distribuidos en diferentes tablas (`users`, `bookings`, `payments`) que deben relacionarse correctamente siguiendo el flujo *Usuario → Reserva → Pago*.
* Existen registros incompletos o inconsistentes en las transacciones que pueden distorsionar los resultados si se analizan directamente.
* Cierta información de eventos viene agrupada en estructuras anidadas (`struct`), lo que dificulta su consulta directa.

**Qué debe soportar el entorno y la infraestructura**
Para resolver esto, necesitamos un entorno en Databricks capaz de soportar un flujo de procesamiento ordenado bajo una arquitectura **Medallion (Bronze, Silver y Gold)** utilizando **Delta Lake**:
1. **Ingesta parametrizada y trazable (Bronze):** El entorno debe recibir los datos originales sin alterarlos, añadiendo metadatos de control (`_ingesta_ts` y `_origen`) y utilizando variables de catálogo, esquema y volumen para facilitar su mantenimiento.
2. **Limpieza y adaptación (Silver):** Debe permitir filtrar registros inválidos, aplanar estructuras anidadas para facilitar su lectura y adaptarse si la estructura de origen cambia con el tiempo (*Schema Evolution*).
3. **Respaldo histórico (Time Travel):** La infraestructura debe conservar versiones anteriores de las tablas para auditar cambios o recuperar datos en caso de errores durante el procesamiento.
4. **Integración analítica (Gold):** Finalmente, debe permitir cruzar las tablas limpias mediante PySpark para entregar datos confiables que respondan directamente a la pregunta de negocio.

---
## 2. Descripción de los datos

La infraestructura almacenará información estructurada de usuarios, reservas y pagos de Wanderbricks, manteniendo las relaciones entre estas entidades para facilitar su procesamiento y análisis. El volumen inicial corresponde a 124.509 registros de `users`, 72.250 de `bookings` y 49.638 de `payments`, por lo que se espera un crecimiento progresivo a medida que se generen nuevos usuarios, reservas y pagos.

La actualización de los datos dependerá de la generación de nuevas reservas y transacciones en la plataforma. Para este proyecto se plantea un proceso de actualización periódico que permita incorporar los nuevos registros y mantener la información disponible para el análisis.

La información será consumida principalmente por analistas de datos y responsables del negocio, quienes podrán utilizarla para consultar indicadores, analizar el comportamiento de las reservas y los métodos de pago, y apoyar la toma de decisiones basada en los datos.

---
## 3. Decisiones de diseño y justificación

### 3.1 Diagrama de la arquitectura

El flujo de datos sigue una arquitectura Medallion sobre Delta Lake, dividiendo las responsabilidades técnicas entre el proveedor de la plataforma (Databricks) y nuestro equipo de trabajo:
Etapa del flujo,Componentes en Wanderbricks,Administrado por el Equipo,Administrado por el Proveedor (Databricks)
1. Fuentes,"Tablas users, bookings, payments y eventos de clickstream.","Selección de tablas y definición de claves de negocio (user_id, booking_id).",Disponibilidad del catálogo de muestra (samples.wanderbricks).
2. Ingesta,Paso inicial hacia la capa Bronce.,"Parametrización (CATALOGO, ESQUEMA, VOLUMEN) y metadatos (_ingesta_ts, _origen).",Conectores de lectura y asignación de memoria para la carga.
3. Almacenamiento,"Tablas en Bronce, Plata y Oro.","Diseño de las capas, control de versiones (Time Travel) y evolución de esquema (Schema Evolution).",Almacenamiento físico en la nube y motor transaccional Delta Lake.
4. Procesamiento,Refinamiento en Plata e integración en Oro.,"Reglas de limpieza en PySpark, aplanado de struct y cruces (join).",Motor distribuido Apache Spark y escalado automático de cómputo.
5. Consumo,Consultas finales de negocio.,"Análisis de métodos de pago (credit_card, paypal, etc.) para responder la pregunta de negocio.",Entorno de visualización del notebook (display) y ejecución de consultas.

* **Qué administra el proveedor (Databricks):** Se encarga de toda la base tecnológica: aprovisionamiento de máquinas, red, almacenamiento físico en la nube, configuración del motor de Spark y disponibilidad del servicio.
* **Qué administra el equipo:** Nos encargamos del diseño de las capas (Bronce, Plata y Oro), la ingesta parametrizada, las reglas de limpieza (como el filtrado de pagos inválidos y el aplanado de estructuras `struct`), el control de versiones (*Time Travel*), la adaptación de esquemas (*Schema Evolution*) y las consultas finales para responder la pregunta de negocio.

---

### 3.2 Matriz de roles

En un entorno real, no todos los usuarios deben tener los mismos permisos sobre las tablas. La siguiente matriz define qué acciones puede realizar cada perfil dentro del catálogo para proteger la calidad y seguridad de los datos:

| Rol | Bronce | Plata | Oro |
|---|---|---|---|
| **Analista** | **Sin acceso.** Los datos están en bruto y contienen registros incompletos o estructuras anidadas que pueden llevar a errores de interpretación. | **Solo lectura (`SELECT`).** Puede explorar las tablas limpias (`users_plata`, `bookings_plata`, `payments_plata`) para validaciones puntuales. | **Lectura (`SELECT`).** Es su capa principal de trabajo para consultar datos integrados y analizar el método de pago predominante. |
| **Ingeniero de datos** | **Lectura y escritura (`SELECT`, `INSERT`, `CREATE`).** Ejecuta la ingesta desde las fuentes originales y agrega los metadatos de trazabilidad. | **Lectura, escritura y modificación.** Aplica reglas de limpieza, aplana estructuras de clickstream y gestiona la evolución del esquema. | **Lectura y escritura.** Construye las tablas finales relacionando usuarios, reservas y pagos mediante operaciones `join`. |
| **Administrador** | **Control total (`ALL PRIVILEGES`).** Configura el catálogo, esquemas y volúmenes, y asigna permisos de ingesta. | **Control total (`ALL PRIVILEGES`).** Supervisa las políticas de acceso, retención de historial (*Time Travel*) y gobierno de datos. | **Control total (`ALL PRIVILEGES`).** Administra quién puede consultar los resultados de negocio y audita el uso de las tablas. |

---

### 3.3 Especificación del equivalente IaaS

Si en lugar de usar Databricks tuviéramos que construir esta misma plataforma desde cero alquilando infraestructura básica en la nube (IaaS, como máquinas virtuales en AWS EC2 o Azure VM), necesitaríamos diseñar y configurar el siguiente entorno:

* **Máquinas y dimensionamiento:** Un clúster pequeño distribuido compuesto por **1 nodo maestro (*Driver*)** de 4 vCPUs y 16 GB de RAM para coordinar las tareas, y **2 nodos trabajadores (*Workers*)** de 4 vCPUs y 16 GB de RAM cada uno para procesar en paralelo las tablas de usuarios, reservas, los 49.638 registros iniciales de pagos y los eventos de clickstream.
* **Sistema operativo:** Linux Ubuntu Server LTS en todos los nodos para garantizar estabilidad y compatibilidad con las herramientas de procesamiento.
* **Software a instalar:** Instalación manual en cada máquina de Java (OpenJDK), Python, Apache Spark, las librerías de código abierto de Delta Lake, un gestor de recursos como Apache YARN y un servidor de JupyterLab para poder escribir y ejecutar el código en notebooks.
* **Red:** Configuración de una red privada virtual (VPC) con una subred interna para que los nodos del clúster se comuniquen entre sí de forma rápida y segura, además de reglas de firewall para restringir el acceso externo únicamente al puerto del entorno de notebooks y administración por SSH.
* **Almacenamiento:** Discos de estado sólido (SSD) en cada máquina virtual para el sistema operativo y archivos temporales de procesamiento, conectados a un servicio de almacenamiento en la nube (como Amazon S3 o Azure Blob Storage) donde se guardarían físicamente los archivos de las capas Bronce, Plata y Oro.
* **Estimación del esfuerzo:**
  * **Puesta en marcha (Alta):** Tomaría varios días de trabajo técnico antes de poder ver un solo dato. Habría que crear las máquinas, conectar la red, instalar Spark y Delta Lake en cada nodo y probar que todo funcione sin fallos de conexión o permisos.
  * **Operación y mantenimiento (Media - Alta):** El equipo tendría que dedicar tiempo constante a encender y apagar máquinas para no gastar presupuesto, actualizar el sistema operativo, revisar errores si un nodo se apaga y gestionar manualmente las copias de seguridad.

---

### 3.4 Comparación IaaS / PaaS / SaaS

| Criterio | IaaS (Infraestructura como Servicio) | PaaS (Plataforma como Servicio) | SaaS / Serverless (Software como Servicio) |
|---|---|---|---|
| **Control** | **Alto.** El equipo controla desde el sistema operativo y la red hasta la versión exacta de cada librería instalada. | **Medio.** El proveedor gestiona los servidores base, pero el equipo configura el tamaño de los clústeres y el entorno de ejecución. | **Bajo sobre la infraestructura, alto sobre los datos.** El entorno ya viene configurado y listo para usar. |
| **Tiempo hasta el primer resultado** | **Lento (días o semanas).** No se puede analizar nada hasta terminar de instalar y conectar toda la infraestructura. | **Rápido (horas).** Solo requiere configurar el entorno de datos y encender el clúster para empezar a trabajar. | **Inmediato (minutos).** Permite conectarse a las tablas de origen y ejecutar código desde el primer momento. |
| **Esfuerzo operativo** | **Alto.** Requiere administrar servidores, parches de seguridad, redes y fallos de hardware. | **Medio - Bajo.** El proveedor mantiene el software actualizado, aunque el equipo aún debe vigilar el uso de los clústeres. | **Mínimo.** El proveedor administra automáticamente los recursos de cómputo y el mantenimiento técnico. |
| **Costo** | **Pago por máquinas encendidas.** Si los servidores quedan prendidos sin usarse, se sigue pagando por ellos. | **Pago por uso de plataforma y cómputo.** Se paga por el tiempo que el clúster permanece activo procesando datos. | **Pago por consumo real (o capa gratuita limitada).** Los recursos se asignan solo cuando se ejecuta una consulta. |
| **Escalabilidad** | **Manual o compleja.** Si los datos crecen, hay que agregar y configurar nuevas máquinas en el clúster. | **Semiautomática.** Permite ajustar el número de nodos desde un panel de control según la carga de trabajo. | **Automática.** La plataforma ajusta los recursos internamente de forma transparente según la consulta. |
| **Gobierno** | **Manual.** Hay que instalar y configurar herramientas adicionales para controlar permisos y auditoría. | **Integrado.** Incluye herramientas propias para gestionar accesos, catálogos y seguridad en un solo lugar. | **Totalmente integrado.** El control de catálogos, esquemas, permisos y trazabilidad viene listo para usar desde el inicio. |

**Conclusión:** 
Para el caso de Wanderbricks, **conviene utilizar un esquema PaaS / SaaS gestionado como el que ofrece Databricks**. Nuestro objetivo principal no es administrar servidores ni configurar redes, sino procesar los datos de usuarios, reservas y pagos de forma confiable para responder qué métodos de pago prefieren los clientes. Al delegar la administración de la infraestructura al proveedor, pudimos concentrar todo el esfuerzo desde el primer día en diseñar la arquitectura Medallion

---
## 4. Implementación
### 4.1 Organización del entorno

In [0]:
# TODO: catálogos, esquemas y volúmenes organizados con criterio

# 1. Variables de parametrización del entorno
CATALOGO = "bigdata_grupo15"
ESQUEMA = "wanderbricks"
VOLUMEN = "datos_crudos"

# 2. Definir el catálogo y esquema activos por defecto
spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql(f"USE SCHEMA {ESQUEMA}")

# 3. Evidencia de la organización del catálogo, tablas (Bronce y Plata) y volumen
print(f"Entorno activo: {CATALOGO}.{ESQUEMA} | Volumen: {VOLUMEN}")

display(spark.sql(f"SHOW SCHEMAS IN {CATALOGO}"))
display(spark.sql(f"SHOW VOLUMES IN {CATALOGO}.{ESQUEMA}"))
display(spark.sql(f"SHOW TABLES IN {CATALOGO}.{ESQUEMA}"))

### 4.2 Permisos

*Al menos dos sentencias GRANT con niveles distintos sobre objetos distintos.*

In [0]:
# Tabla de Plata donde aplicaremos el permiso
TABLA = f"{CATALOGO}.{ESQUEMA}.payments_plata"

# GRANT 1 — a quién, qué y por qué:
# A los usuarios de la cuenta, permiso para usar el esquema wanderbricks y poder ver su contenido.
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {CATALOGO}.{ESQUEMA} TO `account users`")

# GRANT 2 — a quién, qué y por qué:
# A los usuarios de la cuenta, permiso de solo lectura en payments_plata para consultar datos sin modificarlos.
spark.sql(f"GRANT SELECT ON TABLE {TABLA} TO `account users`")

# Mostrar los permisos asignados
display(spark.sql(f"SHOW GRANTS ON SCHEMA {CATALOGO}.{ESQUEMA}"))
display(spark.sql(f"SHOW GRANTS ON TABLE {TABLA}"))

### 4.3 Linaje

A continuación se evidencia el linaje de la tabla `payments_plata` desde el Catalog Explorer. El gráfico muestra la trazabilidad del dato a través de la arquitectura Medallion: partiendo de la fuente original (`samples.wanderbricks.payments`), pasando por la ingesta en la capa Bronce (`payments_bronce`) y llegando finalmente a la tabla limpia en la capa Plata (`payments_plata`).

[Linaje de payments_plata]![image_1790813008671.png](./image_1790813008671.png "image_1790813008671.png")

### 4.4 Automatización

*Un Job con al menos dos tareas encadenadas y una programación definida.
Insertar la captura de una ejecución exitosa e indicar el identificador del Job.*

# TODO: código de las tareas que ejecuta el Job
%md
### 4.4 Automatización

Se configuró un Job en Databricks Workflows para automatizar la validación secuencial de las capas implementadas en el proyecto:

* **Nombre del Job:** `Automatizacion_Wanderbricks`
* **Identificador del Job (ID trabajo):** `616036990504234`
* **Tareas encadenadas:**
  1. `1_Validar_Bronce`: Verifica la disponibilidad y conteo de los registros ingresados en la capa Bronce (49.638 registros en pagos).
  2. `2_Validar_Plata`: Se ejecuta una vez finalizada con éxito la primera tarea y valida los registros limpios en la capa Plata (45.912 registros en pagos).
* **Programación:** Ejecución automática diaria (cómputo Serverless).

[Ejecución exitosa del Job Automatizacion_Wanderbricks]!![4ea02f8d-7791-4ef6-8667-cc64d1f3b06a.png](./4ea02f8d-7791-4ef6-8667-cc64d1f3b06a.png "4ea02f8d-7791-4ef6-8667-cc64d1f3b06a.png")

---
## 5. Resultados


A partir del despliegue, gobierno y automatización de la infraestructura en Databricks para el caso de **Wanderbricks**, obtuvimos los siguientes resultados clave:

* **Organización y trazabilidad del entorno:** La estructura implementada sobre el catálogo `bigdata_grupo15` y el esquema `wanderbricks`, junto con el volumen `datos_crudos`, permitió separar con claridad el estado de los datos. El seguimiento del linaje confirma de extremo a extremo cómo las tablas de origen (`users`, `bookings`, `payments`) pasan por la capa Bronce (conservando metadatos de ingesta como `_ingesta_ts` y `_origen`) hasta llegar a las tablas depuradas en la capa Plata (`users_plata`, `bookings_plata`, `payments_plata`).
* **Protección de la calidad del dato mediante permisos:** En la capa Plata se logró depurar la tabla de pagos pasando de **49.638 registros originales** a **45.912 registros válidos** (descartando **3.726 registros**, equivalentes al **7,51 %**). Al aplicar permisos diferenciados con `GRANT` (`USE SCHEMA` a nivel de esquema y `SELECT` de solo lectura sobre `payments_plata`) y definir la matriz de roles, garantizamos que los analistas consuman únicamente datos limpios para estudiar los métodos de pago (`credit_card`, `paypal`, `apple_pay`, `google_pay`, `bank_transfer`) sin riesgo de alterar las tablas ni consultar registros descartados en Bronce.
* **Confiabilidad operativa mediante automatización:** Con la configuración del Job `Automatizacion_Wanderbricks` (`ID: 616036990504234`) y el encadenamiento de las tareas `1_Validar_Bronce` → `2_Validar_Plata` bajo una programación diaria en cómputo Serverless, el flujo dejó de depender de ejecuciones manuales. Si ocurre algún problema en la capa inicial, la segunda tarea no se ejecuta, evitando que datos inconsistentes lleguen al análisis de negocio.
* **Valor del enfoque gestionado (PaaS/SaaS) frente a IaaS:** El contraste con el diseño equivalente en IaaS demostró que levantar y mantener manualmente un clúster distribuido de Apache Spark en máquinas virtuales habría consumido gran parte del tiempo en tareas de infraestructura (redes, instalación de software y administración de servidores). Utilizar un entorno gestionado con Delta Lake nos permitió disponer de forma inmediata de gobierno unificado, historial de versiones (*Time Travel*), adaptación de estructura (*Schema Evolution*) y automatización, concentrando todo el esfuerzo en asegurar que la relación *Usuario → Reserva → Pago* responda con precisión a la pregunta de negocio.

---
## 6. Conclusiones

A partir de los resultados obtenidos en el despliegue, gobierno y automatización del entorno para **Wanderbricks**, establecemos las siguientes conclusiones:

* **Qué funcionó:**
  * **La separación por capas para proteger el análisis de negocio:** Mantener los datos originales intactos en la capa Bronce (49.638 registros en pagos con sus metadatos `_ingesta_ts` y `_origen`) y realizar el filtrado hacia la capa Plata (donde quedaron 45.912 registros válidos tras descartar el 7,51 % de datos inconsistentes) permitió preparar una base confiable para analizar los métodos de pago (`credit_card`, `paypal`, `apple_pay`, `google_pay`, `bank_transfer`) sin perder la historia original.
  * **La automatización encadenada en Serverless:** El Job `Automatizacion_Wanderbricks` (`ID: 616036990504234`) con las tareas `1_Validar_Bronce` y `2_Validar_Plata` funcionó con tiempos de ejecución muy cortos (alrededor de 26 segundos) y aseguró que la validación de la capa Plata dependa siempre del estado correcto de la capa Bronce.
  * **El ahorro de tiempo operativo frente a IaaS:** Trabajar sobre un entorno gestionado evitó días de configuración manual de servidores, redes e instalación de Spark, permitiéndonos concentrar el tiempo en la calidad de los datos y en las reglas de gobierno.

* **Qué no funcionó (o qué limitaciones encontramos):**
  * **Restricción de grupos en la edición gratuita para aplicar roles reales:** Aunque diseñamos una matriz de roles diferenciada para Analistas, Ingenieros de datos y Administradores, al momento de ejecutar las sentencias `GRANT` nos encontramos con la limitación de que Databricks Free Edition solo permite utilizar grupos predeterminados (como `account users`). Esto impidió crear grupos personalizados en la práctica para probar bloqueos reales por perfil de usuario.
  * **Estructura mixta en el catálogo:** En el catálogo `bigdata_grupo15` se crearon esquemas llamados `bronze`, `silver` y `gold`, pero en la práctica las tablas de las distintas capas (`users_bronce`, `users_plata`, `bookings_bronce`, `bookings_plata`, `payments_bronce`, `payments_plata`) terminaron alojadas juntas dentro de un único esquema general llamado `wanderbricks` diferenciadas por sufijos en su nombre.

* **Qué haríamos distinto si empezáramos de nuevo:**
  * **Ubicar cada tabla en su esquema correspondiente desde el inicio:** En lugar de guardar todas las tablas dentro del esquema `wanderbricks` usando sufijos (`_bronce`, `_plata`), aprovecharíamos los esquemas `bronze`, `silver` y `gold` ya creados en `bigdata_grupo15` para alojar allí cada tabla según su capa. Esto haría mucho más limpio el gobierno de datos, ya que podríamos aplicar un solo permiso `GRANT SELECT ON SCHEMA bigdata_grupo15.silver` para los analistas en lugar de otorgar permisos tabla por tabla.
  * **Modularizar el código en notebooks independientes por capa desde la primera entrega:** En vez de desarrollar la exploración, la ingesta y la limpieza en un único archivo extenso, separaríamos desde el primer día un notebook exclusivo para Bronce, otro para Plata y otro para Oro, facilitando su conexión directa en los Jobs de automatización.

# 7 · Reparto del trabajo y uso de IA

| Integrante | De qué se encargó | Qué sustenta en el video |
|---|---|---|
| **Adolfo Jiménez** | Desarrollo integral de la evidencia: definición de la pregunta de negocio, construcción de la tabla `metodos_pago_oro` en Unity Catalog (`bigdata_grupo15.wanderbricks`), trazabilidad de registros entre las capas Bronce, Plata y Oro, análisis del plan de ejecución inicial (`.explain()`), implementación de la consulta optimizada con `approx_count_distinct`, medición de tiempos y redacción de conclusiones. | Sustentación completa del proyecto: lógica y resultados de la capa Oro, validación del linaje entre capas, comparación de los planes físicos antes y después de optimizar, evaluación de tiempos y límites de la técnica aplicada. |

**Uso de asistentes de IA:** 
Durante el desarrollo de esta evidencia se utilizó asistencia de IA como apoyo para revisar la sintaxis de las transformaciones en PySpark (`F.when` y `F.approx_count_distinct`), interpretar los operadores del plan físico arrojados por `.explain()` y dar formato a las tablas comparativas en las celdas Markdown. Toda la construcción de la capa Oro, la verificación de conteos entre capas, la ejecución de las pruebas de rendimiento y la interpretación de los resultados sobre los datos de Wanderbricks fueron realizadas y validadas de forma individual directamente en la plataforma.

---
## 📹 Preguntas obligatorias de sustentación

Cada integrante responde estas tres preguntas en su intervención del video:

1. ¿Qué parte de esta arquitectura administra el proveedor y cuál administran ustedes?
2. Muestre un GRANT que ejecutó y explique a quién le está dando qué, y por qué.
3. Si tuvieran que montar esto sobre máquinas virtuales, ¿qué sería lo primero que se les complicaría?

---
## ✅ Antes de entregar

- [ ] El diagrama de arquitectura está incluido y descrito
- [ ] Los dos GRANT están ejecutados y el SHOW GRANTS muestra el resultado
- [ ] El Job tiene dos o más tareas, está programado y hay evidencia de ejecución
- [ ] La comparación IaaS/PaaS/SaaS termina en una conclusión, no en una tabla suelta
- [ ] El enlace del video está en la portada y abre desde otra cuenta
- [ ] Todo confirmado en /ea2 y el HTML subido a Canvas